# Part B — Fine-Tuning the Best Model on Food101

**Assignment 2 · Computer Vision**

In Part A we trained three frozen-backbone models on Food101. In this notebook we take the **best checkpoint from Part A** and **fine-tune** it by selectively unfreezing layers of the backbone.

## Strategy
1. Load the best pre-trained model from Part A checkpoint
2. Unfreeze a configurable number of the deepest backbone layers (`NUM_LAYERS_TO_UNFREEZE`)
3. Train end-to-end with a **lower learning rate** for the backbone parameters
4. Experiment with at least 2 unfreezing configurations and compare results

| Experiment | Layers Unfrozen | Notes |
|---|---|---|
| Baseline (Part A) | 0 (head only) | Loaded from checkpoint |
| Experiment 1 | Last 2 layer blocks | Shallow fine-tune |
| Experiment 2 | Last 3 layer blocks | Deep fine-tune |

---

## 0 · Imports & Configuration

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models

# PyTorch-Ignite
try:
    from ignite.engine import Engine, Events
    from ignite.metrics import Accuracy
    from ignite.metrics import Loss as IgniteLoss
    from ignite.handlers import EarlyStopping
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pytorch-ignite", "-q"])
    from ignite.engine import Engine, Events
    from ignite.metrics import Accuracy
    from ignite.metrics import Loss as IgniteLoss
    from ignite.handlers import EarlyStopping

import matplotlib.pyplot as plt
import numpy as np

# ── Reproducibility ─────────────────────────────────────────────────────────
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# ── Device ──────────────────────────────────────────────────────────────────
DEVICE = (
    torch.device("cuda") if torch.cuda.is_available() else
    torch.device("mps")  if torch.backends.mps.is_available() else
    torch.device("cpu")
)
print(f"Using device: {DEVICE}")

## 1 · Path & Hyperparameter Configuration

Set `BEST_MODEL_NAME` to the model that achieved the highest accuracy in Part A.
The corresponding checkpoint will be loaded from `CHECKPOINT_DIR`.

### Fine-tuning knobs

| Variable | Description |
|---|---|
| `BEST_MODEL_NAME` | `"googlenet"`, `"mobilenet"`, or `"resnet"` |
| `NUM_LAYERS_TO_UNFREEZE` | Number of top-level backbone children to unfreeze (counting from the end). Set to `0` to train head only. |
| `BACKBONE_LR` | Learning rate for unfrozen backbone params (typically 10× smaller than head LR) |
| `HEAD_LR` | Learning rate for the classification head |
| `NUM_EPOCHS_FT` | Maximum fine-tuning epochs |

In [ ]:
# ── Environment ──────────────────────────────────────────────────────────────
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive", force_remount=False)
    BASE_DIR = Path("/content")
else:
    BASE_DIR = Path(os.path.abspath(""))  # notebook directory

CHECKPOINT_DIR = BASE_DIR / "checkpoints"
DATA_DIR       = BASE_DIR / "data" / "food-101"
IMG_DIR        = DATA_DIR / "images"
META_DIR       = DATA_DIR / "meta"

# ── Best model from Part A ────────────────────────────────────────────────────
# Change this to whichever model scored highest in Part A
BEST_MODEL_NAME = "resnet"          # "googlenet" | "mobilenet" | "resnet"

CHECKPOINT_PATH = CHECKPOINT_DIR / f"{BEST_MODEL_NAME}_best.pth"
assert CHECKPOINT_PATH.exists(), (
    f"Checkpoint not found: {CHECKPOINT_PATH}\n"
    "Please make sure Part A has been run and the checkpoint exists."
)
print(f"Checkpoint: {CHECKPOINT_PATH}")

# ── Fine-tuning hyperparameters ───────────────────────────────────────────────
#
# NUM_LAYERS_TO_UNFREEZE controls how many of the deepest top-level children
# of the backbone are unfrozen.  For example:
#   0  → only the classification head trains (replicates Part A behaviour)
#   2  → last 2 backbone blocks + head  (Experiment 1)
#   3  → last 3 backbone blocks + head  (Experiment 2)
#  -1  → unfreeze EVERYTHING (full fine-tune)
#
# Each experiment cell overrides this value — this is just a shared default.
NUM_LAYERS_TO_UNFREEZE = 2          # default; overridden per experiment

BACKBONE_LR    = 1e-4               # LR for unfrozen backbone params
HEAD_LR        = 1e-3               # LR for the classification head
NUM_EPOCHS_FT  = 10                 # Max fine-tuning epochs
BATCH_SIZE     = 64
NUM_WORKERS    = 2
NUM_CLASSES    = 101
IMG_SIZE       = 224
VAL_FRACTION   = 0.2

IMAGENET_MEAN  = [0.485, 0.456, 0.406]
IMAGENET_STD   = [0.229, 0.224, 0.225]

print(f"Model          : {BEST_MODEL_NAME}")
print(f"Layers unfrozen: {NUM_LAYERS_TO_UNFREEZE}")
print(f"Backbone LR    : {BACKBONE_LR}")
print(f"Head LR        : {HEAD_LR}")

## 2 · Data Loading

We use the same data pipeline as Part A: official Food101 train/test splits read from the metadata files, with a 20 % validation split carved out of the training set.  
Fine-tuning augmentation adds **ColorJitter** and **RandomCrop** on top of the Part A transforms to help the unfrozen layers generalise.

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE + 32, IMG_SIZE + 32)),   # slightly larger, then crop
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# ── Build datasets from official Food101 splits ───────────────────────────────
full_train_ds = datasets.ImageFolder(root=IMG_DIR, transform=train_transform)
full_eval_ds  = datasets.ImageFolder(root=IMG_DIR, transform=eval_transform)

def read_split(filepath):
    with open(filepath) as f:
        return set(line.strip() for line in f)

train_keys = read_split(META_DIR / "train.txt")
test_keys  = read_split(META_DIR / "test.txt")

train_indices, test_indices = [], []
for idx, (fp, _) in enumerate(full_train_ds.samples):
    p   = Path(fp)
    key = f"{p.parent.name}/{p.stem}"
    if key in train_keys:
        train_indices.append(idx)
    elif key in test_keys:
        test_indices.append(idx)

val_size      = int(VAL_FRACTION * len(train_indices))
val_indices   = train_indices[:val_size]
train_indices = train_indices[val_size:]

train_loader = DataLoader(Subset(full_train_ds, train_indices),
                          batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS)
val_loader   = DataLoader(Subset(full_eval_ds,  val_indices),
                          batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader  = DataLoader(Subset(full_eval_ds,  test_indices),
                          batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

print(f"Train : {len(train_indices):,} images")
print(f"Val   : {len(val_indices):,} images")
print(f"Test  : {len(test_indices):,} images")

## 3 · Model Building

### 3.1 Custom head
The same 3-layer head used in Part A is re-created here so the checkpoint weights can be loaded without modification.

### 3.2 Unfreezing utility
`unfreeze_top_n_layers(model, n)` counts the **top-level children** of the backbone (e.g. `layer1`, `layer2`, `layer3`, `layer4` for ResNet) and unfreezes the last `n` of them.  
Setting `n = 0` leaves the backbone frozen; `n = -1` unfreezes the entire backbone.

In [ ]:
def build_custom_head(in_features: int, num_classes: int = 101) -> nn.Sequential:
    """Three fully-connected layer classification head (same as Part A)."""
    return nn.Sequential(
        nn.Linear(in_features, 1024),
        nn.ReLU(inplace=True),
        nn.Dropout(0.3),
        nn.Linear(1024, 512),
        nn.ReLU(inplace=True),
        nn.Dropout(0.3),
        nn.Linear(512, num_classes),
    )


def build_model(model_name: str) -> nn.Module:
    """Re-create the Part A model architecture (frozen backbone + custom head)."""
    if model_name == "googlenet":
        m = models.googlenet(weights=models.GoogLeNet_Weights.IMAGENET1K_V1)
        for p in m.parameters():
            p.requires_grad = False
        m.aux_logits = False
        in_feat = m.fc.in_features
        m.fc    = build_custom_head(in_feat, NUM_CLASSES)

    elif model_name == "mobilenet":
        m = models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.IMAGENET1K_V1)
        for p in m.parameters():
            p.requires_grad = False
        in_feat     = m.classifier[0].in_features
        m.classifier = build_custom_head(in_feat, NUM_CLASSES)

    elif model_name == "resnet":
        m = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
        for p in m.parameters():
            p.requires_grad = False
        in_feat = m.fc.in_features
        m.fc    = build_custom_head(in_feat, NUM_CLASSES)

    else:
        raise ValueError(f"Unknown model: {model_name}")

    return m.to(DEVICE)


def get_backbone_children(model: nn.Module, model_name: str):
    """Return the top-level backbone children (excluding the classification head)."""
    if model_name == "googlenet":
        # Everything except the final fc layer
        return [(n, m) for n, m in model.named_children() if n != "fc"]
    elif model_name == "mobilenet":
        return [(n, m) for n, m in model.named_children() if n != "classifier"]
    elif model_name == "resnet":
        return [(n, m) for n, m in model.named_children() if n != "fc"]
    return []


def unfreeze_top_n_layers(model: nn.Module, model_name: str, n: int) -> None:
    """
    Unfreeze the last `n` top-level backbone children.

    Parameters
    ----------
    model      : the model whose backbone to partially unfreeze
    model_name : "googlenet" | "mobilenet" | "resnet"
    n          : number of deepest children to unfreeze
                 0  → backbone stays frozen
                 -1 → unfreeze all backbone children
    """
    if n == 0:
        print("Backbone fully frozen — training head only.")
        return

    children = get_backbone_children(model, model_name)
    to_unfreeze = children if n == -1 else children[-n:]

    for name, module in to_unfreeze:
        for p in module.parameters():
            p.requires_grad = True
        print(f"  Unfrozen backbone layer: {name}")

    frozen = [n for n, _ in children if n not in {nm for nm, _ in to_unfreeze}]
    print(f"  Still frozen: {frozen}")


def count_trainable(model: nn.Module):
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Trainable params: {trainable:,} / {total:,} "
          f"({100 * trainable / total:.1f} %)")

## 4 · Load Checkpoint & Apply Unfreezing

The Part A checkpoint contains only the **state_dict** of the model (saved with `torch.save(model.state_dict(), ...)`).  
We rebuild the architecture first, load the weights, then selectively unfreeze layers.

In [ ]:
# 1. Rebuild architecture (backbone frozen, custom head attached)
model = build_model(BEST_MODEL_NAME)

# 2. Load Part A weights
state_dict = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
model.load_state_dict(state_dict)
print(f"Loaded weights from {CHECKPOINT_PATH}")

# 3. Unfreeze the desired number of backbone layers
print(f"\nUnfreezing top {NUM_LAYERS_TO_UNFREEZE} backbone layer(s):")
unfreeze_top_n_layers(model, BEST_MODEL_NAME, NUM_LAYERS_TO_UNFREEZE)

# 4. Always keep the head trainable
head_attr = "fc" if BEST_MODEL_NAME in ("googlenet", "resnet") else "classifier"
for p in getattr(model, head_attr).parameters():
    p.requires_grad = True

print()
count_trainable(model)

## 5 · Fine-Tuning Loop

We use a **differential learning rate** strategy:
- Unfrozen backbone params → `BACKBONE_LR` (small, to preserve learned features)
- Classification head params → `HEAD_LR` (larger, head needs more adjustment)

The scheduler reduces both LRs on validation loss plateau.

In [ ]:
def fine_tune(
    model,
    model_name: str,
    n_unfrozen: int,
    train_loader,
    val_loader,
    epochs: int = NUM_EPOCHS_FT,
    early_stop_patience: int = 4,
):
    """
    Fine-tune `model` with differential learning rates.

    Returns
    -------
    history : dict with keys train_loss, val_loss, train_acc, val_acc
    """
    criterion = nn.CrossEntropyLoss()

    head_attr     = "fc" if model_name in ("googlenet", "resnet") else "classifier"
    head_params   = list(getattr(model, head_attr).parameters())
    head_ids      = {id(p) for p in head_params}
    backbone_params = [p for p in model.parameters()
                       if p.requires_grad and id(p) not in head_ids]

    param_groups = [{"params": head_params, "lr": HEAD_LR}]
    if backbone_params:
        param_groups.append({"params": backbone_params, "lr": BACKBONE_LR})

    optimiser = optim.Adam(param_groups, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimiser, mode="min", factor=0.5, patience=3
    )

    use_amp = DEVICE.type == "cuda"
    scaler  = torch.amp.GradScaler("cuda", enabled=use_amp)

    history     = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    best_val_acc = 0.0
    run_label   = f"{model_name}_ft_unfreeze{n_unfrozen}"

    # ── Ignite engines ────────────────────────────────────────────────────────
    def _train_step(engine, batch):
        model.train()
        images, labels = batch
        images, labels = images.to(DEVICE, non_blocking=True), labels.to(DEVICE, non_blocking=True)
        optimiser.zero_grad()
        with torch.amp.autocast(device_type=DEVICE.type, enabled=use_amp):
            outputs = model(images)
            loss    = criterion(outputs, labels)
        scaler.scale(loss).backward()
        scaler.step(optimiser)
        scaler.update()
        engine.state.train_loss_sum = getattr(engine.state, "train_loss_sum", 0.0) + loss.item() * images.size(0)
        engine.state.train_correct  = getattr(engine.state, "train_correct",  0)   + outputs.argmax(1).eq(labels).sum().item()
        engine.state.train_total    = getattr(engine.state, "train_total",    0)   + labels.size(0)
        return outputs, labels

    def _eval_step(engine, batch):
        model.eval()
        with torch.no_grad():
            images, labels = batch
            images, labels = images.to(DEVICE, non_blocking=True), labels.to(DEVICE, non_blocking=True)
            with torch.amp.autocast(device_type=DEVICE.type, enabled=use_amp):
                outputs = model(images)
            return outputs, labels

    trainer   = Engine(_train_step)
    evaluator = Engine(_eval_step)

    IgniteLoss(criterion).attach(evaluator, "val_loss")
    Accuracy().attach(evaluator, "val_acc")

    early_stop = EarlyStopping(
        patience=early_stop_patience,
        score_function=lambda e: -e.state.metrics["val_loss"],
        trainer=trainer,
    )
    evaluator.add_event_handler(Events.COMPLETED, early_stop)

    @trainer.on(Events.EPOCH_COMPLETED)
    def _on_epoch_end(engine):
        nonlocal best_val_acc
        epoch  = engine.state.epoch
        tr_loss = engine.state.train_loss_sum / engine.state.train_total
        tr_acc  = engine.state.train_correct  / engine.state.train_total

        evaluator.run(val_loader)
        val_loss = evaluator.state.metrics["val_loss"]
        val_acc  = evaluator.state.metrics["val_acc"]

        scheduler.step(val_loss)

        history["train_loss"].append(tr_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(tr_acc)
        history["val_acc"].append(val_acc)

        current_lr = optimiser.param_groups[0]["lr"]
        print(f"  Epoch {epoch:2d}/{epochs} | "
              f"train_loss={tr_loss:.4f}  train_acc={tr_acc:.4f} | "
              f"val_loss={val_loss:.4f}  val_acc={val_acc:.4f} | "
              f"lr={current_lr:.2e}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_path    = CHECKPOINT_DIR / f"{run_label}_best.pth"
            torch.save(model.state_dict(), best_path)
            print(f"    ✓ New best val_acc={best_val_acc:.4f} — saved to {best_path}")

        # Reset per-epoch accumulators
        engine.state.train_loss_sum = 0.0
        engine.state.train_correct  = 0
        engine.state.train_total    = 0

    trainer.run(train_loader, max_epochs=epochs)
    return history, best_val_acc

## 6 · Experiment 1 — Unfreeze Last 2 Backbone Layers

We start with a moderate unfreezing: the **two deepest** top-level backbone blocks are unfrozen alongside the head.  
This allows the model to adapt its highest-level feature representations while keeping the majority of pre-trained weights frozen.

In [ ]:
EXP1_LAYERS = 2

# Rebuild and reload for a clean experiment
model_exp1 = build_model(BEST_MODEL_NAME)
model_exp1.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=DEVICE))

print(f"=== Experiment 1: unfreeze last {EXP1_LAYERS} backbone layer(s) ===")
unfreeze_top_n_layers(model_exp1, BEST_MODEL_NAME, EXP1_LAYERS)
head_attr = "fc" if BEST_MODEL_NAME in ("googlenet", "resnet") else "classifier"
for p in getattr(model_exp1, head_attr).parameters():
    p.requires_grad = True
count_trainable(model_exp1)

print()
history_exp1, best_acc_exp1 = fine_tune(
    model_exp1, BEST_MODEL_NAME, EXP1_LAYERS,
    train_loader, val_loader,
)
print(f"\nExperiment 1 best val accuracy: {best_acc_exp1:.4f}")

## 7 · Experiment 2 — Unfreeze Last 3 Backbone Layers

We now open up more of the backbone to gradient updates.  
With more trainable capacity the model can adapt mid-level features to food-specific patterns, but the risk of **catastrophic forgetting** increases — mitigated by the low `BACKBONE_LR`.

In [ ]:
EXP2_LAYERS = 3

model_exp2 = build_model(BEST_MODEL_NAME)
model_exp2.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=DEVICE))

print(f"=== Experiment 2: unfreeze last {EXP2_LAYERS} backbone layer(s) ===")
unfreeze_top_n_layers(model_exp2, BEST_MODEL_NAME, EXP2_LAYERS)
head_attr = "fc" if BEST_MODEL_NAME in ("googlenet", "resnet") else "classifier"
for p in getattr(model_exp2, head_attr).parameters():
    p.requires_grad = True
count_trainable(model_exp2)

print()
history_exp2, best_acc_exp2 = fine_tune(
    model_exp2, BEST_MODEL_NAME, EXP2_LAYERS,
    train_loader, val_loader,
)
print(f"\nExperiment 2 best val accuracy: {best_acc_exp2:.4f}")

## 8 · Learning Curves

Side-by-side plots of training vs. validation loss and accuracy for both experiments.

In [ ]:
def plot_history(histories: dict, title: str = "Fine-Tuning Learning Curves"):
    """
    Parameters
    ----------
    histories : {label: history_dict}  where history_dict has keys
                train_loss / val_loss / train_acc / val_acc
    """
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(title, fontsize=14)

    for label, h in histories.items():
        epochs = range(1, len(h["train_loss"]) + 1)
        axes[0].plot(epochs, h["train_loss"], linestyle="--", label=f"{label} train")
        axes[0].plot(epochs, h["val_loss"],   linestyle="-",  label=f"{label} val")
        axes[1].plot(epochs, h["train_acc"],  linestyle="--", label=f"{label} train")
        axes[1].plot(epochs, h["val_acc"],    linestyle="-",  label=f"{label} val")

    axes[0].set_title("Loss");     axes[0].set_xlabel("Epoch"); axes[0].legend()
    axes[1].set_title("Accuracy"); axes[1].set_xlabel("Epoch"); axes[1].legend()
    plt.tight_layout()
    plt.show()


plot_history({
    f"unfreeze={EXP1_LAYERS}": history_exp1,
    f"unfreeze={EXP2_LAYERS}": history_exp2,
})

## 9 · Test Set Evaluation

Load the **best checkpoint** from each experiment and evaluate on the held-out test set.

In [ ]:
def evaluate_on_test(model_name: str, n_unfrozen: int, test_loader) -> float:
    """Load the best fine-tuned checkpoint and compute test accuracy."""
    run_label  = f"{model_name}_ft_unfreeze{n_unfrozen}"
    ckpt_path  = CHECKPOINT_DIR / f"{run_label}_best.pth"

    m = build_model(model_name)
    unfreeze_top_n_layers(m, model_name, n_unfrozen)
    m.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
    m.eval()

    correct = total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            preds   = m(images).argmax(1)
            correct += preds.eq(labels).sum().item()
            total   += labels.size(0)

    acc = correct / total
    print(f"[{run_label}]  test accuracy = {acc:.4f} ({100*acc:.2f} %)")
    return acc


print("=" * 55)
print("Test Set Results")
print("=" * 55)

# Part A baseline (frozen backbone, already evaluated in Part A — shown for reference)
print(f"Part A baseline  (unfreeze=0) — see Part A notebook")

acc_exp1 = evaluate_on_test(BEST_MODEL_NAME, EXP1_LAYERS, test_loader)
acc_exp2 = evaluate_on_test(BEST_MODEL_NAME, EXP2_LAYERS, test_loader)

## 10 · Comparison Summary

A concise table comparing the two fine-tuning experiments.

In [ ]:
import pandas as pd

results = pd.DataFrame([
    {
        "Experiment": f"Exp 1 — unfreeze {EXP1_LAYERS} layer(s)",
        "Layers Unfrozen": EXP1_LAYERS,
        "Best Val Acc": f"{best_acc_exp1:.4f}",
        "Test Acc": f"{acc_exp1:.4f}",
    },
    {
        "Experiment": f"Exp 2 — unfreeze {EXP2_LAYERS} layer(s)",
        "Layers Unfrozen": EXP2_LAYERS,
        "Best Val Acc": f"{best_acc_exp2:.4f}",
        "Test Acc": f"{acc_exp2:.4f}",
    },
])

print(results.to_string(index=False))

## 11 · Discussion

Fill in this section after running both experiments:

**Which configuration performed better, and why?**

> *[Your analysis here — e.g., did deeper unfreezing help or hurt? Was there evidence of catastrophic forgetting? How did training loss vs. validation loss differ between experiments?]*

**Effect of the differential learning rate**

> *[Comment on whether the low `BACKBONE_LR` was effective at preserving pre-trained features while still allowing the backbone to adapt.]*

**Comparison with Part A**

> *[Compare the test accuracy from the best fine-tuning experiment against the Part A frozen-backbone result. Quantify the improvement (or lack thereof).]*